# Unit 3 — Statistical Baselines: the breakeven horizon

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bgalon/geo-graph-learning/blob/main/unit-3-statistical-baselines/geoai-graph-unit3.ipynb)

**Lesson context.** Before anyone reaches for a deep spatio-temporal network, they owe the world a *baseline*: the simplest forecast that already works. A baseline is the **spec for "better"** — it is the number every fancy model must beat to justify its complexity. This demo dramatizes the **breakeven question**: forecasting freeway speed at **one** loop-detector from only its own past, *how far ahead can a SARIMA model beat a dumb historical average — and what does it miss because it cannot see its upstream neighbor?*

**What you'll build.** On **METR-LA** (LA County freeway speeds, 2012) you'll (1) load one sensor's speed series and *see* its daily/weekly seasonality; (2) test **stationarity** (ADF) and read **ACF/PACF**, then make the series stationary with regular + seasonal differencing; (3) fit **one transparent SARIMAX** in the open (Box–Jenkins loop + residual diagnostics) and meet the **Kalman filter** as its online cousin; (4) run an honest **walk-forward** evaluation across the 15/30/60-min horizons and find the **breakeven horizon** where SARIMA degrades to the historical-average floor — *the lab deliverable*; (5) load an **upstream** sensor and compute its **lagged cross-correlation** with the target, making the **spatial gap** visible — the bridge to Unit 5's graph model.

**Prerequisites.** pandas time-series fluency (a `DatetimeIndex`, resampling); mean / variance / correlation; from **U1** the graph + segment + CRS mental model; from **U2** the Lagrangian (per-vehicle) → Eulerian (per-sensor) framing — METR-LA is the Eulerian view.

**Data.** METR-LA — Li, Yu, Shahabi & Liu (2018), *DCRNN* ([arXiv:1707.01926](https://arxiv.org/abs/1707.01926)). This notebook is **self-contained**: it fetches the full archive from a public mirror, **cuts the two sensor columns it needs in-cell**, and caches the slim slice — no pre-staged file, no separate prep script (a curated parquet on the course Drive is the fallback). U5 inherits the *same* dataset.

---

> **© 2026 Ben Galon. All rights reserved.** Part of the Geo-AI course (The Arena). Provided to enrolled students for course use; not for redistribution.
>
> **AI-assisted materials.** These notebooks were drafted with AI assistance (Claude Code) using curated course context, and reviewed by the instructor. They are learning references, not guaranteed-correct keys — verify before you rely on them.

### About the sensors — what METR-LA actually measures, and where

**The instrument.** METR-LA's 207 sensors are **inductive-loop detectors** — wire coils embedded in the freeway pavement that sense each vehicle passing over them. They belong to **Caltrans PeMS**, the California **Performance Measurement System**, which aggregates the raw detector pulses into **5-minute** summaries of **speed (mph)**, flow (veh/hr), and occupancy. METR-LA keeps the **speed** channel for LA County freeways over **Mar–Jun 2012** (the integer column names are **PeMS station ids**).

- **Official source + live data + field documentation:** Caltrans PeMS — <https://pems.dot.ca.gov/>
- **Dataset provenance:** Li, Yu, Shahabi & Liu (2018), *DCRNN* ([arXiv:1707.01926](https://arxiv.org/abs/1707.01926)); station ids and coordinates come from the DCRNN `graph_sensor_locations.csv`.

**The two stations this demo uses** (just south-east of downtown LA):

| role | PeMS station | location (lat, lon) | why it was chosen |
|---|---|---|---|
| **target** | `773012` | 34.0839, −118.2209 | freeway-mainline detector with a clean daily peak/off-peak cycle |
| **upstream** | `760650` | 34.0751, −118.2326 | ~1.5 km upstream on the same corridor; its speed **leads** the target by ~10 min |

You'll see their exact positions — and the flow arrow between them — on the **interactive map in Section 8a**. The key intuition for Beat 5: a loop detector is a **point** sensor. It knows its own patch of asphalt intimately and **nothing** about the road 1.5 km upstream — which is exactly the blind spot a graph model (Unit 5) is built to close.

## 1. Setup

Fetch the shared Colab helper and install this unit's pinned requirements. On Colab this `pip install`s the unit-3 forecasting stack; locally (a `uv` env) it is a no-op.

In [ ]:
# Fetch the shared setup helper and install this unit's pinned requirements.
import os, urllib.request

if not os.path.exists("setup_colab.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/bgalon/geo-graph-learning/main/setup_colab.py",
        "setup_colab.py",
    )

from setup_colab import setup_unit
setup_unit("unit-3")

## 2. Smoke test — fail fast before any teaching content

Import every major library, do one trivial call each, **and warm the numba JIT** with a tiny `ARIMA` fit so the first real fit in Section 5 is not paying the one-time compile cost. On failure this prints one clean message pointing to `KNOWN_ISSUES.md` (raised *outside* the `except` per the IPython-9 rule).

In [ ]:
# Smoke test: imports + one trivial call each + warm the statsforecast numba JIT.
_smoke_error = None
try:
    import numpy as np
    import pandas as pd
    import matplotlib
    import matplotlib.pyplot as plt
    import statsmodels
    from statsmodels.tsa.stattools import adfuller
    import statsforecast
    from statsforecast import StatsForecast
    from statsforecast.models import ARIMA, HistoricAverage, SeasonalNaive
    import folium
    import gdown
    import h5py

    # Trivial calls.
    _ = pd.Series([1.0, 2.0, 3.0])
    _ = np.arange(3)
    _ = adfuller                 # importable (used in Section 3)
    _ = gdown.download           # importable (fallback data fetch)
    _ = h5py.File                # importable (primary data fetch cuts the h5)
    _ = folium.Map()             # importable (Section 6 map)

    # Warm the numba JIT: a tiny ARIMA fit on 30 synthetic points. The FIRST
    # statsforecast fit compiles numba kernels (~seconds); we pay that cost here,
    # behind the setup talk, not in the timed walk-forward loop.
    _warm = pd.DataFrame({
        "unique_id": "warm",
        "ds": pd.date_range("2012-01-01", periods=30, freq="15min"),
        "y": (60 + 5 * np.sin(np.arange(30) / 3.0)).astype(float),
    })
    _sf = StatsForecast(models=[ARIMA(order=(1, 1, 1), seasonal_order=(1, 1, 1), season_length=4)], freq="15min")
    _ = _sf.forecast(df=_warm, h=2)

    print("numpy       ", np.__version__)
    print("pandas      ", pd.__version__)
    print("matplotlib  ", matplotlib.__version__)
    print("statsmodels ", statsmodels.__version__)
    print("statsforecast", statsforecast.__version__)
    print("folium      ", folium.__version__)
    print("h5py        ", h5py.__version__)
except Exception as e:
    _smoke_error = e

# Raise OUTSIDE the except (no chaining) + `from None`, so a failure is ONE clean
# message, not IPython's nested-traceback noise on Colab (py3.12 / IPython 9.x).
if _smoke_error is not None:
    print("\n" + "=" * 70)
    print("SMOKE TEST FAILED:", _smoke_error)
    print("Do NOT continue — fix the environment first.")
    print("- If the setup cell above printed a WARN/404 for requirements/unit-3.txt,")
    print("  the unit deps never installed. Re-run setup once that file is published")
    print("  (see unit-3-statistical-baselines/KNOWN_ISSUES.md).")
    print("- If the error mentions numba/statsforecast on Windows, see the")
    print("  'statsforecast / numba Windows wheel' entry in KNOWN_ISSUES.md.")
    print("=" * 70)
    raise RuntimeError(f"Smoke test failed: {_smoke_error}") from None

print("\nSmoke test passed - environment is ready (numba JIT warm).")

## 3. Configuration — sensors, aggregation, horizons (one place to switch)

Everything tunable lives here so the demo is a **parameter change, not a rewrite**.

**Sensor ids (PINNED).** `TARGET_SENSOR = 773012` / `UPSTREAM_SENSOR = 760650` were chosen by this heuristic on the real METR-LA data: rank sensors by **within-day-profile variance**, take a mid-pack freeway-mainline detector with **high lag-1-day autocorrelation** (a clean daily peak/off-peak + weekday/weekend cycle), then confirm a **physically upstream** neighbor whose speed *leads* the target with a **cross-correlation peak at +5–15 min**. Here 760650 sits ~1.5 km upstream and leads 773012 by ~10 min (r≈0.58). Both ids are columns the loader cuts straight from the dataset.

**Aggregation + season (the clock-vs-fidelity knob).** The **safe default** is 15-min aggregation, daily season `m = 96`, headline horizons **15/30/60 min** — this keeps the walk-forward loop comfortably inside the 45-min budget. To switch to native 5-min (all four 5/15/30/60-min horizons, heavier fits), change the three values noted inline. The instructor times *both* configs on a Colab Run-All and picks the default that fits the 45-min budget.

In [ ]:
# ── Sensor selection (PINNED via the heuristic, on real METR-LA data) ───────
# Chosen by the slice-gen heuristic: 773012 is a freeway-mainline detector with a
# clean daily cycle (lag-1-day autocorr ~0.57, ~93% complete); 760650 is a
# physically-upstream neighbor (~1.5 km) whose speed LEADS the target by ~10 min
# (lagged cross-corr ~0.58). These are real METR-LA station ids and ARE the two
# columns the loader cuts from the dataset.
TARGET_SENSOR   = "773012"     # freeway-mainline target sensor
UPSTREAM_SENSOR = "760650"     # physically-upstream neighbor (~10 min lead)

# ── Aggregation + seasonality (SAFE DEFAULT = 15-min, daily season) ──────────
AGG       = "15min"            # native is "5min". SWITCH HERE for 5-min fidelity.
SEASON_M  = 96                 # daily season: 96 steps @15min. Use 288 for "5min".
HORIZONS  = [15, 30, 60]       # minutes. Use [5, 15, 30, 60] for the "5min" config.
#
# To run NATIVE 5-min instead (heavier; time it on Colab first), set:
#     AGG = "5min";  SEASON_M = 288;  HORIZONS = [5, 15, 30, 60]
# Nothing else changes — h, step_size, and the horizon→step map below are derived.

STEP_MIN  = int(AGG.replace("min", ""))          # minutes per step (5 or 15)
H_STEPS   = max(HORIZONS) // STEP_MIN            # forecast length covering the max horizon
HORIZON_STEPS = {h: h // STEP_MIN for h in HORIZONS}   # horizon(min) -> step index

print(f"target={TARGET_SENSOR}  upstream={UPSTREAM_SENSOR}")
print(f"AGG={AGG}  SEASON_M={SEASON_M}  HORIZONS(min)={HORIZONS}")
print(f"step={STEP_MIN}min  h={H_STEPS} steps  horizon->step {HORIZON_STEPS}")

## 4. Beat 1 — The hook: one sensor, and the floor it must beat

**Concept.** Load a single sensor's speed series; *see* the recurrent daily/weekly structure; meet the **naive** and **historical-average** baselines — the floor every model must beat.

The notebook is **self-contained**: the next cell fetches the full METR-LA archive from a public mirror and **cuts just the two columns we need in-cell** (target + upstream), then caches the slim slice as parquet so re-runs are instant. A small curated parquet on the course Drive is the fallback. Either way we end with a clean `DatetimeIndex` and treat **0 mph as missing** (next cell).

In [ ]:
# ── Inline data acquisition: fetch once, CUT the two columns in-cell, cache. ─
# Self-contained (CLAUDE.md): a fresh Colab runs this end-to-end with nothing
# pre-staged. PRIMARY = the full METR-LA HDF5 from a public GitHub raw mirror
# (CDN-backed, no per-file rate limit in a full class) — we read it and cut just
# the target + upstream columns. FALLBACK = a small curated 2-column parquet on
# the course Google Drive (a fast backup once its id is wired). First success
# wins; the slim 2-column result is cached as parquet for instant re-runs.
import os
import pandas as pd

# Cache target (Colab uses /content; local uses cwd).
PARQUET_PATH = "/content/metr_la_demo.parquet" if os.path.isdir("/content") \
    else "metr_la_demo.parquet"

# PRIMARY: full metr-la.h5 on a public GitHub raw mirror (~57 MB, [34272, 207],
# real station-id columns + a 5-min DatetimeIndex). Self-contained, no Drive.
METR_LA_H5_URL = (
    "https://raw.githubusercontent.com/deepkashiwa20/MegaCRN/main/METRLA/metr-la.h5"
)
# FALLBACK: small curated 2-column slice on the course Google Drive. Optional fast
# backup — wire the file id after uploading metr_la_demo.parquet (Anyone-with-link).
METR_LA_DRIVE_ID = "REPLACE_WITH_DRIVE_FILE_ID"


def _subset_from_h5(h5_path) -> pd.DataFrame:
    """Read the fixed-format METR-LA h5 with h5py and cut the two sensor columns.

    Using h5py directly (not pd.read_hdf) sidesteps the pandas/pytables version
    fragility called out in KNOWN_ISSUES.md; h5py is pre-installed on Colab.
    """
    import h5py
    want = [TARGET_SENSOR, UPSTREAM_SENSOR]
    with h5py.File(h5_path, "r") as f:
        cols = [c.decode() if isinstance(c, bytes) else str(c) for c in f["/df/axis0"][:]]
        ci = {c: k for k, c in enumerate(cols)}
        missing = [c for c in want if c not in ci]
        if missing:
            raise KeyError(f"sensor(s) {missing} not in METR-LA columns")
        idx = pd.to_datetime(f["/df/axis1"][:])
        vals = f["/df/block0_values"][:, [ci[TARGET_SENSOR], ci[UPSTREAM_SENSOR]]]
    return pd.DataFrame(vals, index=idx, columns=want)


def _load_curated_slice() -> pd.DataFrame:
    if os.path.exists(PARQUET_PATH) and os.path.getsize(PARQUET_PATH) > 0:
        print(f"using cached slice: {PARQUET_PATH}")
        return pd.read_parquet(PARQUET_PATH)

    errors = []
    # PRIMARY — full h5 from the GitHub raw mirror, cut to two columns in-cell.
    try:
        import urllib.request
        h5_local = "/content/metr-la.h5" if os.path.isdir("/content") else "metr-la.h5"
        if not (os.path.exists(h5_local) and os.path.getsize(h5_local) > 0):
            print("fetching full METR-LA h5 from the public mirror (primary)…")
            urllib.request.urlretrieve(METR_LA_H5_URL, h5_local)
        df = _subset_from_h5(h5_local)
        df.to_parquet(PARQUET_PATH)   # cache the slim 2-column slice
        print(f"cut columns {list(df.columns)} from the full archive → {PARQUET_PATH}")
        return df
    except Exception as exc:  # noqa: BLE001
        errors.append(f"GitHub h5: {exc}")

    # FALLBACK — small curated 2-column parquet on Google Drive.
    try:
        import gdown
        if METR_LA_DRIVE_ID and not METR_LA_DRIVE_ID.startswith("REPLACE"):
            print("primary failed — fetching curated slice from Google Drive (fallback)…")
            out = gdown.download(id=METR_LA_DRIVE_ID, output=PARQUET_PATH, quiet=False)
            if out and os.path.exists(PARQUET_PATH):
                return pd.read_parquet(PARQUET_PATH)
            errors.append("gdown returned no file")
        else:
            errors.append("Drive: METR_LA_DRIVE_ID not wired (still a placeholder)")
    except Exception as exc:  # noqa: BLE001
        errors.append(f"Drive: {exc}")

    print("\n" + "=" * 70)
    print("Could not load METR-LA from any source:")
    for e in errors:
        print("  -", e)
    print("Check network access to raw.githubusercontent.com, or wire")
    print("METR_LA_DRIVE_ID to the backup slice (see KNOWN_ISSUES.md).")
    print("=" * 70)
    raise RuntimeError("METR-LA load failed (all sources exhausted).") from None


raw = _load_curated_slice()

# Ensure a clean, sorted DatetimeIndex named for clarity.
if not isinstance(raw.index, pd.DatetimeIndex):
    raw.index = pd.to_datetime(raw.index)
raw = raw.sort_index()
raw.index.name = "timestamp"

print(raw.shape)
raw.head()

In [ ]:
# Pull the target + upstream columns, name the zero-as-missing gotcha, aggregate.
# The slice is ALREADY pre-masked (zeros -> NaN), but we SHOW students the check:
# in raw METR-LA, 0 mph means "sensor missing", not "stopped traffic". Forgetting
# this makes every stationarity diagnostic lie.
import numpy as np

# Defensive mask in case a raw column slips through (no-op on the clean slice).
clean = raw[[TARGET_SENSOR, UPSTREAM_SENSOR]].replace(0.0, np.nan)

print("missing fraction (should be small after masking):")
print((clean.isna().mean()).round(4))

# Aggregate to the chosen cadence (mean speed per AGG bin); interpolate short gaps
# so the regularly-spaced series the models need has no holes.
series = clean.resample(AGG).mean().interpolate(limit=SEASON_M // 4, limit_direction="both")
target = series[TARGET_SENSOR]
upstream = series[UPSTREAM_SENSOR]

print(f"\naggregated to {AGG}: {len(series)} timesteps, "
      f"{series.index.min()} -> {series.index.max()}")
series.head()

### 4a. See the recurrent structure — line + weekday/weekend profile + a fingerprint heatmap

*What to notice:* the **same shape repeats every day** — that repetition is exactly what a dumb seasonal baseline already captures, and the bar SARIMA must clear.

In [ ]:
# V2a — a 2-week line plot of the target series, weekends shaded.
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

two_weeks = target.loc[target.index[0]: target.index[0] + pd.Timedelta(days=14)]

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(two_weeks.index, two_weeks.values, color="#1f78b4", lw=0.9)
# Shade weekend spans.
for day in pd.date_range(two_weeks.index[0].normalize(), two_weeks.index[-1], freq="D"):
    if day.weekday() >= 5:  # Sat/Sun
        ax.axvspan(day, day + pd.Timedelta(days=1), color="#fdbf6f", alpha=0.25)
ax.set(xlabel="time", ylabel="speed (mph)",
       title=f"Target sensor {TARGET_SENSOR}: two weeks of speed "
             f"(weekends shaded) — the daily commute cycle repeats")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d"))
plt.tight_layout()
plt.show()

In [ ]:
# V2b — the mean weekly profile WITH a ±1.5 std band: average speed by
# time-of-day, weekday vs weekend. THE MEAN CURVE *IS* the historical-average
# baseline we race SARIMA against; the band shows how variable each moment is.
prof = target.to_frame("speed").copy()
prof["tod"] = prof.index.hour + prof.index.minute / 60.0
prof["is_weekend"] = prof.index.weekday >= 5


def _profile(mask):
    g = prof[mask].groupby("tod")["speed"]
    mu, sd = g.mean(), g.std()
    lo = (mu - 1.5 * sd).clip(lower=0)   # speed can't go negative
    hi = mu + 1.5 * sd
    return mu, lo, hi


wk_m, wk_lo, wk_hi = _profile(~prof["is_weekend"])
we_m, we_lo, we_hi = _profile(prof["is_weekend"])

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(wk_m.index, wk_m.values, color="#e31a1c", lw=2, label="weekday mean")
ax.fill_between(wk_lo.index, wk_lo.values, wk_hi.values, color="#e31a1c",
                alpha=0.15, label="weekday ±1.5 std")
ax.plot(we_m.index, we_m.values, color="#33a02c", lw=2, label="weekend mean")
ax.fill_between(we_lo.index, we_lo.values, we_hi.values, color="#33a02c",
                alpha=0.15, label="weekend ±1.5 std")
ax.set(xlabel="hour of day", ylabel="speed (mph)",
       title="Mean weekly profile (the historical-average forecast) with a "
             "±1.5 std band — the rush-hour dip is also the most variable",
       xlim=(0, 24), xticks=range(0, 25, 3))
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# V2c — the seasonality "fingerprint": mean speed over hour-of-day x day-of-week.
# One tile that makes the daily + weekly structure pop in a single image.
fp = target.to_frame("speed").copy()
fp["hour"] = fp.index.hour
fp["dow"] = fp.index.dayofweek  # 0=Mon
grid = fp.groupby(["dow", "hour"])["speed"].mean().unstack("hour")

fig, ax = plt.subplots(figsize=(12, 3.6))
im = ax.imshow(grid.values, aspect="auto", cmap="RdYlGn", origin="lower")
ax.set(xlabel="hour of day", ylabel="day of week",
       title="Speed fingerprint: rush-hour dips (dark) carve a weekday block; "
             "weekends stay green")
ax.set_yticks(range(7))
ax.set_yticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
ax.set_xticks(range(0, 24, 3))
fig.colorbar(im, ax=ax, label="mean speed (mph)")
plt.tight_layout()
plt.show()

### 4b. Decompose the structure — trend + daily + weekly + noise

*What to notice:* the series is **mostly seasonality**. **MSTL** (Multiple-Season-Trend decomposition by LOESS) peels the target into a slow **trend**, a **daily** cycle, a **weekly** cycle, and a **remainder** (noise). The daily + weekly bands are exactly the structure a seasonal model — or the historical-average baseline — already captures; what's left in the remainder is the genuinely hard, unpredictable part. This also previews Beat 2: there is **no runaway trend** here (freeway speed is physically bounded), so the work is all in the *seasonal* terms, not in removing a drift.

In [ ]:
# Decompose the target into trend + daily + weekly seasonality + remainder (MSTL).
# MSTL handles MULTIPLE seasonal periods at once (statsmodels >=0.14); the periods
# scale with AGG so this works for the 15-min default and the 5-min config alike.
from statsmodels.tsa.seasonal import MSTL

y_dec = target.interpolate(limit_direction="both").dropna()
periods = (SEASON_M, SEASON_M * 7)            # (daily, weekly) in steps
mstl = MSTL(y_dec, periods=periods).fit()

daily = mstl.seasonal.iloc[:, 0]              # period = SEASON_M
weekly = mstl.seasonal.iloc[:, 1]             # period = SEASON_M * 7
comps = [("observed", y_dec, "#1f78b4"),
         ("trend", mstl.trend, "#e31a1c"),
         ("daily season", daily, "#ff7f00"),
         ("weekly season", weekly, "#33a02c"),
         ("remainder (noise)", mstl.resid, "#6a3d9a")]

# Plot a two-week window so each component is legible (not 4 months squashed).
win = slice(y_dec.index[0], y_dec.index[0] + pd.Timedelta(days=14))
fig, axes = plt.subplots(len(comps), 1, figsize=(13, 11), sharex=True)
for ax, (name, s, c) in zip(axes, comps):
    ax.plot(s.loc[win].index, s.loc[win].values, color=c, lw=0.9)
    ax.set_ylabel(name, fontsize=9)
axes[0].set_title("MSTL decomposition: trend + daily + weekly + noise "
                  "(two-week window) — the structure SARIMA must encode")
axes[-1].set_xlabel("time")
plt.tight_layout()
plt.show()

# How much of the signal does each component explain? (variance share)
parts = {"trend": mstl.trend, "daily": daily, "weekly": weekly, "noise": mstl.resid}
tot = sum(np.nanvar(v.values) for v in parts.values())
print("share of variance by component:")
for name, v in parts.items():
    print(f"  {name:>7}: {np.nanvar(v.values) / tot:5.1%}")

**The two baselines, in domain vocabulary.**
- **Naive:** next value = last value. Cheap; good for one step, useless across a rush-hour transition.
- **Historical average / seasonal-naive:** next value = the mean at the *same time-of-day* in prior cycles — literally the red/green profile above. This is the **specific floor** the breakeven question targets: *can SARIMA beat the long-run daily pattern, and for how far ahead?*

## 5. Beat 2 — Stationarity as a contract

**Concept.** ARIMA wants a **stationary** series — constant mean/variance, no trend, no season. Two different things can break that: a **stochastic trend** (a *unit root* — the series wanders like a random walk) and **seasonality** (a repeating cycle). They need different fixes, and — crucially — **different tests catch them.**

The **ADF test targets only the unit root.** Its null hypothesis is "there *is* a unit root (non-stationary)." Freeway speed is physically **bounded and mean-reverting** (it can't wander off to 200 mph), so we actually expect ADF to **reject** the null here — i.e. *no stochastic trend*. **But ADF is blind to seasonality:** the giant daily cycle you just saw is still there. So passing ADF does **not** mean "ready for ARIMA" — we still read the **ACF/PACF** and apply a **seasonal difference** to strip the daily cycle. *Library: statsmodels (the transparent path).*

In [ ]:
# ADF on the RAW series. Null = "unit root" (non-stationary). Speed is bounded &
# mean-reverting, so we expect a SMALL p => REJECT the null => no stochastic trend.
from statsmodels.tsa.stattools import adfuller

raw_y = target.dropna()
adf_stat, adf_p, *_ = adfuller(raw_y, autolag="AIC")
print(f"ADF (raw):  statistic={adf_stat:.3f}   p-value={adf_p:.4g}")
verdict = "reject null => NO unit root (mean-reverting)" if adf_p < 0.05 \
    else "fail to reject => unit root (stochastic trend)"
print(f"=> {verdict}.")
print("BUT ADF says nothing about seasonality: the daily cycle is still in there")
print("(see the ACF below). We seasonally difference to remove it before ARIMA.")

### 5a. Show, don't print — the before/after differencing panel

*What to notice:* on the raw series the rolling **mean is roughly flat** (consistent with "no stochastic trend" from ADF) but it rides a strong **periodic swing** every day; the rolling **std pulses** with the rush-hour cycle. After a regular + **seasonal** difference, both collapse toward a flat, noise-like band — the daily cycle has been removed. The ADF numbers ride in the titles.

In [ ]:
# Regular first difference (d=1) + seasonal difference at the daily lag (D=1, m=SEASON_M).
# The SEASONAL difference is the one that matters here (it strips the daily cycle);
# the regular difference is the conventional ARIMA d=1 companion.
diff_y = raw_y.diff(1).diff(SEASON_M).dropna()
adf_stat_d, adf_p_d, *_ = adfuller(diff_y, autolag="AIC")
print(f"ADF (differenced): statistic={adf_stat_d:.3f}   p-value={adf_p_d:.4g}")
print("Still strongly stationary (very negative stat), and — more to the point —")
print("the daily seasonal structure is now removed (see the differenced ACF below).")

In [ ]:
# V3a — rolling mean/std, raw (periodic swings) vs differenced (flat). ADF in titles.
win = SEASON_M  # one day

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=False)

axes[0].plot(raw_y.index, raw_y.values, color="#1f78b4", lw=0.6, label="speed")
axes[0].plot(raw_y.index, raw_y.rolling(win).mean(), color="#e31a1c", lw=1.5,
             label="rolling mean (1 day)")
axes[0].plot(raw_y.index, raw_y.rolling(win).std(), color="#ff7f00", lw=1.2,
             label="rolling std (1 day)")
axes[0].set(ylabel="mph",
            title=f"RAW — flat mean (no trend) but strong daily swings  "
                  f"(ADF p={adf_p:.3g}: no unit root, yet still seasonal)")
axes[0].legend(loc="lower left", fontsize=8)

axes[1].plot(diff_y.index, diff_y.values, color="#6a3d9a", lw=0.5, label="differenced")
axes[1].plot(diff_y.index, diff_y.rolling(win).mean(), color="#e31a1c", lw=1.5,
             label="rolling mean")
axes[1].plot(diff_y.index, diff_y.rolling(win).std(), color="#ff7f00", lw=1.2,
             label="rolling std")
axes[1].set(ylabel="d mph", xlabel="time",
            title=f"DIFFERENCED (1)(1)_m — daily cycle removed, rolling stats flat  "
                  f"(ADF p={adf_p_d:.3g})")
axes[1].legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

### 5b. Read the autocorrelation — ACF/PACF before vs after

*What to notice:* this is the plot that **exposes the seasonality ADF missed.** On the raw series the ACF doesn't die away — it stays high and **re-peaks at the daily lag** (a slow, periodic decay), the signature of a strong seasonal cycle. After the seasonal difference the ACF **collapses toward zero** and the daily-lag spike is gone. This is also where you *read off* candidate `p`/`q` orders and confirm the seasonal period.

In [ ]:
# V3b — ACF/PACF, raw (top) vs differenced (bottom), daily-season lag annotated.
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

nlags = min(SEASON_M + 12, len(raw_y) // 2 - 1)

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
plot_acf(raw_y, lags=nlags, ax=axes[0, 0], title="ACF — raw (slow, periodic decay)")
plot_pacf(raw_y, lags=nlags, ax=axes[0, 1], title="PACF — raw", method="ywm")
plot_acf(diff_y, lags=nlags, ax=axes[1, 0], title="ACF — differenced (collapses)")
plot_pacf(diff_y, lags=nlags, ax=axes[1, 1], title="PACF — differenced", method="ywm")
for ax in axes.ravel():
    ax.axvline(SEASON_M, color="#33a02c", ls="--", lw=1)
    ax.text(SEASON_M, ax.get_ylim()[1] * 0.8, " daily\n lag",
            color="#33a02c", fontsize=8)
plt.tight_layout()
plt.show()

**When the contract breaks (seeds the wrong-class lesson).** Stationarity is an assumption, not a fact. **Holidays, accidents, weather, and route changes** all violate it — the mean/variance shift and a model fit on "normal" days mis-forecasts. Naming these now is the honest setup for *choosing a different model class* when the data stops behaving (practice extension (d)).

## 6. Beat 3 — The ARIMA family + Kalman: one explained fit

**Concept.** `ARIMA(p,d,q)` + seasonal `(P,D,Q,m)` = **SARIMA**. We fit *one* in the open via the **Box–Jenkins loop** — *identification* (the differencing/ACF above) → *estimation* (MLE) → *diagnostic checking* (residuals should be white noise) — then meet the **Kalman filter** as the online cousin. *Library: statsmodels (one fully-explained fit).*

We fit on a **short, recent window** so the slow pure-Python optimizer stays inside the clock.

In [ ]:
# One transparent SARIMAX fit on a short recent window (keeps the optimizer fast).
from statsmodels.tsa.statespace.sarimax import SARIMAX

WINDOW_DAYS = 7
_t = target.dropna()
# Last WINDOW_DAYS of data (Series.last() is deprecated in pandas 2.x -> use a mask).
short = _t.loc[_t.index.max() - pd.Timedelta(days=WINDOW_DAYS):]

# Orders read off Section 5: regular (1,1,1), seasonal (1,1,1, m=SEASON_M).
# Kept deliberately small so this single fit is demo-fast, not production-tuned.
sarimax = SARIMAX(
    short,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, SEASON_M),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
res = sarimax.fit(disp=False)
print(res.summary())

### 6a. Diagnostic checking — is the residual white noise?

*What to notice:* good residuals look like **noise** — no autocorrelation (flat residual-ACF), roughly normal (the Q–Q line), constant spread. Structure left in the residuals means the model missed something. This is the metric-gets-a-distribution-plot beat for fit quality.

In [ ]:
# V4a — the 4-panel residual diagnostic (the white-noise check).
fig = res.plot_diagnostics(figsize=(12, 7))
fig.suptitle("SARIMAX residual diagnostics — looking for white noise", y=1.01)
plt.tight_layout()
plt.show()

# The Ljung-Box test puts a number on "are residuals uncorrelated?"
from statsmodels.stats.diagnostic import acorr_ljungbox
lb = acorr_ljungbox(res.resid.dropna(), lags=[SEASON_M], return_df=True)
print("Ljung-Box at the daily lag (large p => residuals look uncorrelated):")
print(lb)

In [ ]:
# V4b — in-sample fit overlaid on the actual short window (see it before trusting it).
fitted = res.fittedvalues
plot_from = short.index[SEASON_M + 1]  # skip the differencing warm-up region

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(short.loc[plot_from:].index, short.loc[plot_from:].values,
        color="#1f78b4", lw=1.2, label="actual")
ax.plot(fitted.loc[plot_from:].index, fitted.loc[plot_from:].values,
        color="#e31a1c", lw=1.0, ls="--", label="SARIMAX fitted")
ax.set(xlabel="time", ylabel="speed (mph)",
       title="SARIMAX in-sample fit tracks the daily cycle on the training window")
ax.legend()
plt.tight_layout()
plt.show()

### 6b. Kalman sidebar — the online cousin of ARIMA

A **Kalman filter** keeps a hidden *state* and updates it as each new observation arrives: predict → observe → correct, forever. ARIMA fits a fixed model to a batch; the Kalman filter is the **online** version — the production transit baseline (e.g. *TheTransitClock*), and the bridge to the **bus-archive practice**. In fact statsmodels fits SARIMAX *through* a state-space Kalman filter under the hood, so you have already used one. Below is the one-line gesture: the filtered (online) estimate. *(Cut first if the clock is tight.)*

In [ ]:
# Kalman gesture: SARIMAX is fit *through* a state-space Kalman filter, so its
# one-step-ahead in-sample predictions ARE the filter's online estimate — each
# point uses only data up to the previous step. We surface them WITH their
# uncertainty band (the running estimate ± the filter's 95% interval).
pred = res.get_prediction()            # in-sample, one-step-ahead (non-dynamic)
kf_mean = pred.predicted_mean
kf_ci = pred.conf_int(alpha=0.05)      # two columns: lower / upper

fig, ax = plt.subplots(figsize=(13, 3.6))
ax.plot(short.loc[plot_from:].index, short.loc[plot_from:].values,
        color="#1f78b4", lw=1.0, label="actual")
ax.plot(kf_mean.loc[plot_from:].index, kf_mean.loc[plot_from:].values,
        color="#6a3d9a", lw=1.0, label="Kalman filtered (online) estimate")
_lo = kf_ci.iloc[:, 0].loc[plot_from:]
_hi = kf_ci.iloc[:, 1].loc[plot_from:]
ax.fill_between(_lo.index, _lo.values, _hi.values, color="#6a3d9a", alpha=0.15,
                label="95% interval")
ax.set_xlim(plot_from, short.index[-1])
ax.set(xlabel="time", ylabel="speed (mph)",
       title="The Kalman filter updates its estimate as each observation arrives "
             "(online cousin of ARIMA)")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Beat 4 — Honest evaluation: walk-forward + the breakeven horizon

**Concept.** A single train/test split flatters the model. **Rolling-origin / walk-forward** evaluation slides the forecast origin forward and averages error over many origins — honest because it mimics forecasting in production. We score **SARIMA, HistoricAverage, and SeasonalNaive in the SAME call** (identical protocol = fair), report **RMSE/MAE/MAPE** per horizon, and find the **breakeven horizon** — the first horizon where SARIMA's error rises to meet the historical-average floor. *That crossing is the lab deliverable.* *Library: statsforecast `ARIMA` (numba-JIT) — the fixed orders from Beats 2-3, fit on a capped recent window so the loop stays well inside the class budget.*

In [ ]:
# Reshape to statsforecast tidy format: unique_id, ds, y.
sf_df = (
    target.dropna()
    .rename("y")
    .reset_index()
    .rename(columns={"timestamp": "ds"})
)
sf_df["unique_id"] = TARGET_SENSOR
sf_df = sf_df[["unique_id", "ds", "y"]]
print(f"tidy frame: {len(sf_df)} rows; freq={AGG}; season_length={SEASON_M}")
sf_df.head()

In [ ]:
# The walk-forward loop — the one cost center, now sized to run fast on free Colab.
# Two speed-ups that DON'T change the model we identified in Beats 2-3:
#   1) Use the FIXED orders (1,1,1)(1,1,1)_m instead of AutoARIMA's stepwise
#      SEARCH. The search — rerun for every origin — was the real cost; we already
#      read these orders off the ACF/PACF, so we just fit them directly.
#   2) Cap each fit's training history to the last INPUT_DAYS via `input_size`
#      (the same "fit on a recent window" choice as the single SARIMAX fit above):
#      a daily-seasonal model needs a few weeks, not 4 months, of history.
from statsforecast import StatsForecast
from statsforecast.models import ARIMA, HistoricAverage, SeasonalNaive

N_WINDOWS = 8
STEP_SIZE = H_STEPS                   # non-overlapping origins, one max-horizon apart
INPUT_DAYS = 7
INPUT_SIZE = SEASON_M * INPUT_DAYS    # train on the last INPUT_DAYS only, per origin

sf = StatsForecast(
    models=[
        # SARIMA with the orders identified in Beats 2-3 (fixed => no search).
        ARIMA(order=(1, 1, 1), seasonal_order=(1, 1, 1), season_length=SEASON_M),
        HistoricAverage(),                       # the long-run-mean floor
        SeasonalNaive(season_length=SEASON_M),   # same-time-yesterday floor (cheap)
    ],
    freq=AGG,
    n_jobs=1,
)

cv = sf.cross_validation(
    df=sf_df,
    h=H_STEPS,
    step_size=STEP_SIZE,
    n_windows=N_WINDOWS,
    input_size=INPUT_SIZE,            # <- key speed-up: bounded training window per fit
)
print(cv.shape)
cv.head()

In [ ]:
# Score RMSE/MAE/MAPE per horizon (5/15/30/60 min) by slicing the forecast path.
# Each cv row carries the step's lead via (ds - cutoff); map that to a horizon.
import numpy as np

cvx = cv.reset_index() if cv.index.name else cv.copy()
cvx["lead_min"] = ((cvx["ds"] - cvx["cutoff"]).dt.total_seconds() / 60).round().astype(int)

MODEL_COLS = {"ARIMA": "SARIMA", "HistoricAverage": "HistoricAvg",
              "SeasonalNaive": "SeasonalNaive"}


def _metrics(g, pred_col):
    err = g[pred_col] - g["y"]
    rmse = float(np.sqrt((err ** 2).mean()))
    mae = float(err.abs().mean())
    # MAPE is fragile near zero speeds — pre-masking helps, but guard anyway.
    denom = g["y"].replace(0, np.nan)
    mape = float((err.abs() / denom).dropna().mean() * 100)
    return rmse, mae, mape


rows = []
for h_min in HORIZONS:
    lead = h_min  # lead_min equals the horizon in minutes
    gh = cvx[cvx["lead_min"] == lead]
    for raw_col, nice in MODEL_COLS.items():
        if raw_col not in gh.columns:
            continue
        rmse, mae, mape = _metrics(gh, raw_col)
        rows.append({"horizon_min": h_min, "model": nice,
                     "RMSE": rmse, "MAE": mae, "MAPE": mape})

scores = pd.DataFrame(rows)
scores_wide = scores.pivot(index="horizon_min", columns="model", values="RMSE")
print("RMSE by horizon (mph):")
scores.pivot_table(index="horizon_min", columns="model",
                   values=["RMSE", "MAE", "MAPE"]).round(2)

In [ ]:
# Find the breakeven horizon programmatically: first horizon where SARIMA RMSE
# rises to meet (>=) the historical-average floor.
breakeven = None
for h_min in HORIZONS:
    s = scores_wide.loc[h_min, "SARIMA"]
    f = scores_wide.loc[h_min, "HistoricAvg"]
    if s >= f:
        breakeven = h_min
        break

if breakeven is None:
    print(f"SARIMA beats the historical-average floor across ALL tested horizons "
          f"({HORIZONS} min) — breakeven is beyond {max(HORIZONS)} min.")
else:
    print(f"BREAKEVEN HORIZON = {breakeven} min: at/after this lead, SARIMA no "
          f"longer beats the dumb historical-average floor.")

### 7a. The headline plot — error vs. horizon, with the breakeven marked

*What to notice:* **left of the line SARIMA earns its complexity; right of it the dumb floor is just as good.** This single chart *is* the lab answer.

In [ ]:
# V5a — error vs horizon, SARIMA vs HistoricAverage (+ SeasonalNaive), breakeven line.
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(scores_wide.index, scores_wide["SARIMA"], "o-", color="#e31a1c",
        lw=2, label="SARIMA (ARIMA)")
ax.plot(scores_wide.index, scores_wide["HistoricAvg"], "s--", color="#1f78b4",
        lw=2, label="Historical average (the floor)")
if "SeasonalNaive" in scores_wide.columns:
    ax.plot(scores_wide.index, scores_wide["SeasonalNaive"], "^:",
            color="#33a02c", lw=1.5, label="Seasonal naive")
if breakeven is not None:
    ax.axvline(breakeven, color="#6a3d9a", ls="-", lw=1.5)
    ax.annotate(f"breakeven ~{breakeven} min",
                xy=(breakeven, ax.get_ylim()[1] * 0.9),
                xytext=(8, 0), textcoords="offset points",
                color="#6a3d9a", fontweight="bold")
ax.set(xlabel="forecast horizon (minutes ahead)", ylabel="RMSE (mph)",
       title="How far ahead does SARIMA beat the floor? — the breakeven question",
       xticks=HORIZONS)
ax.legend()
plt.tight_layout()
plt.show()

### 7b. Not just means — the error *distribution* at each horizon

*What to notice:* the breakeven is a story about **spreads**, not just two mean points. Side-by-side boxplots guard against "the means crossed but the variance tells a different story."

In [ ]:
# V5b — side-by-side per-window error boxplots across horizons (SARIMA vs floor).
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(4 * len(HORIZONS), 4.2),
                         sharey=True)
if len(HORIZONS) == 1:
    axes = [axes]
for ax, h_min in zip(axes, HORIZONS):
    gh = cvx[cvx["lead_min"] == h_min]
    data, labels, colors = [], [], []
    for raw_col, nice, col in [("ARIMA", "SARIMA", "#e31a1c"),
                               ("HistoricAverage", "HistAvg", "#1f78b4")]:
        if raw_col in gh.columns:
            data.append((gh[raw_col] - gh["y"]).abs().values)
            labels.append(nice)
            colors.append(col)
    bp = ax.boxplot(data, labels=labels, patch_artist=True, showfliers=False)
    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.5)
    ax.set(title=f"{h_min} min", xlabel="model")
axes[0].set_ylabel("|forecast error| (mph)")
fig.suptitle("Absolute-error distribution per horizon — SARIMA's edge shrinks "
             "as the horizon grows", y=1.02)
plt.tight_layout()
plt.show()

### 7c. The qualitative *why* — forecast vs. actual for one window

*What to notice:* SARIMA tracks the short-term wiggle, then **reverts toward the seasonal mean** at longer leads — which is exactly where the flat historical-average floor catches up.

In [ ]:
# V5c — one walk-forward window: actual vs SARIMA path vs flat HistoricAverage.
last_cut = cvx["cutoff"].max()
w = cvx[cvx["cutoff"] == last_cut].sort_values("ds")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(w["ds"], w["y"], "o-", color="#1f78b4", lw=1.5, label="actual")
if "ARIMA" in w.columns:
    ax.plot(w["ds"], w["ARIMA"], "s--", color="#e31a1c", lw=1.5,
            label="SARIMA forecast")
if "HistoricAverage" in w.columns:
    ax.plot(w["ds"], w["HistoricAverage"], "d:", color="#33a02c", lw=1.5,
            label="historical-average forecast")
ax.axvline(last_cut, color="#999999", ls="-", lw=1)
ax.text(last_cut, ax.get_ylim()[0], " forecast origin", color="#666666", fontsize=8)
ax.set(xlabel="time", ylabel="speed (mph)",
       title=f"One forecast window: SARIMA tracks then reverts toward the floor "
             f"over the {max(HORIZONS)}-min horizon")
ax.legend()
plt.tight_layout()
plt.show()

### 7d. See the forecast itself — a dynamic multi-step prediction with uncertainty

*What to notice:* the aggregate RMSE hides the *shape* of a forecast. Here SARIMA predicts **6 hours ahead from a single origin**: the point forecast plus **80% / 95% prediction-interval bands** that **widen with the horizon** — the honest statistical statement *"confident now, less sure later."* The two baselines are overlaid **with their own ranges**: the **historical average** (a flat line + a constant band) and the **seasonal-naive** (yesterday's curve). Where SARIMA's band is *tighter* than a baseline's and still brackets the actual, the model is genuinely adding information; where the bands overlap heavily, the extra complexity isn't buying much — the same breakeven story, now visible step by step.

In [ ]:
# 7d — A dynamic multi-step forecast with prediction-interval bands (the "fan"),
# the baselines overlaid WITH their own ranges. We fit at a cutoff a few days
# before the series end (so there is real future to judge against) and at a
# morning hour so the 6-h window spans the rush-hour transition. statsforecast
# gives numba-fast PI bands via level=[...] — the SAME fixed-order model as the loop.
from statsforecast import StatsForecast
from statsforecast.models import ARIMA, HistoricAverage, SeasonalNaive

H_FAN = 6 * 60 // STEP_MIN                          # forecast 6 hours ahead
_end = target.dropna().index[-1]
cutoff_ts = (_end - pd.Timedelta(days=4)).normalize() + pd.Timedelta(hours=6)  # ~6am
train = target.loc[cutoff_ts - pd.Timedelta(days=INPUT_DAYS):cutoff_ts].dropna()
train_df = train.rename("y").reset_index().rename(columns={"timestamp": "ds"})
train_df["unique_id"] = TARGET_SENSOR
train_df = train_df[["unique_id", "ds", "y"]]

future_idx = pd.date_range(cutoff_ts + pd.Timedelta(minutes=STEP_MIN),
                           periods=H_FAN, freq=AGG)
actual_future = target.reindex(future_idx)

sf_fan = StatsForecast(
    models=[ARIMA(order=(1, 1, 1), seasonal_order=(1, 1, 1), season_length=SEASON_M),
            HistoricAverage(), SeasonalNaive(season_length=SEASON_M)],
    freq=AGG, n_jobs=1,
)
fan = sf_fan.forecast(df=train_df, h=H_FAN, level=[80, 95])
fx = fan["ds"]

fig, ax = plt.subplots(figsize=(13, 5))
# Context (last 12 h of training) + the actual future.
ctx = train.loc[cutoff_ts - pd.Timedelta(hours=12):]
ax.plot(ctx.index, ctx.values, color="#999999", lw=1.0, label="history (train)")
ax.plot(actual_future.index, actual_future.values, color="#1f78b4", lw=2.2,
        label="actual (what really happened)")

# SARIMA point forecast + widening 80% / 95% prediction-interval fan.
ax.plot(fx, fan["ARIMA"], color="#e31a1c", lw=2, label="SARIMA forecast")
ax.fill_between(fx, fan["ARIMA-lo-95"], fan["ARIMA-hi-95"], color="#e31a1c",
                alpha=0.12, label="SARIMA 95% PI")
ax.fill_between(fx, fan["ARIMA-lo-80"], fan["ARIMA-hi-80"], color="#e31a1c",
                alpha=0.22, label="SARIMA 80% PI")

# Baselines, each WITH its own 95% range (lighter) so the uncertainty is comparable.
ax.plot(fx, fan["HistoricAverage"], color="#33a02c", ls="--", lw=1.6,
        label="historical average")
ax.fill_between(fx, fan["HistoricAverage-lo-95"], fan["HistoricAverage-hi-95"],
                color="#33a02c", alpha=0.08)
ax.plot(fx, fan["SeasonalNaive"], color="#6a3d9a", ls=":", lw=1.8,
        label="seasonal naive (yesterday)")
ax.fill_between(fx, fan["SeasonalNaive-lo-95"], fan["SeasonalNaive-hi-95"],
                color="#6a3d9a", alpha=0.08)

ax.axvline(cutoff_ts, color="#000000", lw=1)
ax.text(cutoff_ts, ax.get_ylim()[0], " forecast origin", color="#444444", fontsize=8)
ax.set(xlabel="time", ylabel="speed (mph)",
       title=f"SARIMA {H_FAN * STEP_MIN // 60}-h forecast with 80/95% prediction "
             f"intervals, vs the baselines' ranges")
ax.legend(fontsize=8, ncol=2, loc="lower left")
plt.tight_layout()
plt.show()

## 8. Beat 5 — The spatial gap: the missing neighbor

**Concept.** What can *one* sensor never see? Its **neighbors**. We load the **upstream** sensor (already in the slice) and compute its **lagged cross-correlation** with the target. A peak at a **positive lag (~5–15 min)** means upstream speed *now* predicts target speed *soon* — real signal in the system, **invisible to the univariate SARIMA**. That is the budget Unit 5's graph model is asked to recover.

In [ ]:
# V6a — lagged cross-correlation: corr(upstream_{t-k}, target_t) over k=0..~20 min.
import numpy as np

paired = pd.concat([target.rename("tgt"), upstream.rename("ups")], axis=1).dropna()
max_lag_min = 30
max_lag = max_lag_min // STEP_MIN

lags_min, corrs = [], []
for k in range(0, max_lag + 1):
    c = paired["ups"].shift(k).corr(paired["tgt"])
    lags_min.append(k * STEP_MIN)
    corrs.append(c)
corrs = np.array(corrs)
peak_i = int(np.nanargmax(corrs))
peak_lag, peak_corr = lags_min[peak_i], corrs[peak_i]

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.stem(lags_min, corrs, basefmt=" ")
ax.axvline(peak_lag, color="#e31a1c", ls="--", lw=1.5)
ax.annotate(f"peak at +{peak_lag} min (r={peak_corr:.2f})",
            xy=(peak_lag, peak_corr), xytext=(10, -10),
            textcoords="offset points", color="#e31a1c", fontweight="bold")
ax.set(xlabel="lag k (minutes): upstream leads target by k",
       ylabel="correlation",
       title="Upstream speed LEADS the target — signal SARIMA can't see")
plt.tight_layout()
plt.show()
print(f"Upstream leads the target by ~{peak_lag} min (cross-corr peak r={peak_corr:.2f}).")

### 8a. The geographic anchor — an interactive map of the two sensors

*What to notice:* "upstream" is a **real place on the road**, not an abstraction. The faded nearby sensors preview Unit 5's sensor graph; the snapshot colors show speed *at one instant*; the arrow carries the lead time you just measured. Toggle basemaps (incl. a blank topology-only layer).

In [ ]:
# V6b — interactive folium map: target + upstream markers, lead-time arrow,
# faded nearby sensors (U5 graph teaser), and a "speed now" color snapshot.
import folium

# Sensor lat/lon from METR-LA's graph_sensor_locations.csv (pinned at slice-gen).
SENSOR_LATLON = {
    TARGET_SENSOR:   (34.08390, -118.22086),   # 773012 — freeway-mainline target
    UPSTREAM_SENSOR: (34.07505, -118.23256),   # 760650 — upstream neighbor (~10 min lead)
}
# Faded nearby sensors (U5 graph teaser) — the closest other detectors to the
# target, from graph_sensor_locations.csv. They preview Unit 5's sensor graph.
NEARBY_LATLON = [
    (34.07787, -118.22871),   # 771673
    (34.07828, -118.22834),   # 772669
    (34.06712, -118.23973),   # 718045
    (34.07314, -118.23388),   # 771667
    (34.08374, -118.22076),   # 773013
]

# A "speed now" snapshot at one timestamp, colored by speed (green=fast, red=slow).
snap_t = paired.index[len(paired) // 2]
snap = {TARGET_SENSOR: target.get(snap_t, np.nan),
        UPSTREAM_SENSOR: upstream.get(snap_t, np.nan)}


def _speed_color(v):
    if v != v:  # NaN
        return "#888888"
    return "#33a02c" if v >= 55 else "#ff7f00" if v >= 35 else "#e31a1c"


t_lat, t_lon = SENSOR_LATLON[TARGET_SENSOR]
u_lat, u_lon = SENSOR_LATLON[UPSTREAM_SENSOR]
center = [(t_lat + u_lat) / 2, (t_lon + u_lon) / 2]

_ESRI = ("https://server.arcgisonline.com/ArcGIS/rest/services/"
         "World_Imagery/MapServer/tile/{z}/{y}/{x}")
_BLANK = ("data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwC"
          "AAAAC0lEQVR42mNk+A8AAQUBAScY42YAAAAASUVORK5CYII=")

m = folium.Map(location=center, zoom_start=13, tiles=None, control_scale=True)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
folium.TileLayer(tiles=_ESRI, attr="Tiles © Esri", name="Esri satellite",
                 overlay=False, control=True).add_to(m)
folium.TileLayer(tiles=_BLANK, attr="blank", name="Blank (topology only)",
                 overlay=False, control=True).add_to(m)

# Faded nearby sensors — the U5 graph density teaser.
nearby_fg = folium.FeatureGroup(name="Nearby sensors (U5 graph teaser)")
for la, lo in NEARBY_LATLON:
    folium.CircleMarker([la, lo], radius=4, color="#666666", weight=1,
                        fill=True, fill_color="#bbbbbb", fill_opacity=0.4,
                        tooltip="nearby sensor").add_to(nearby_fg)
nearby_fg.add_to(m)

# Upstream -> target arrow, annotated with the measured lead time. We use folium's
# PolyLineTextPath-free approach: a thick line + a midpoint chevron marker, both
# stable across folium versions (no RegularPolygonMarker dependency).
folium.PolyLine([(u_lat, u_lon), (t_lat, t_lon)], color="#6a3d9a", weight=4,
                opacity=0.85,
                tooltip=f"flow direction: upstream leads target by ~{peak_lag} min"
                ).add_to(m)
mid_lat, mid_lon = (u_lat + t_lat) / 2, (u_lon + t_lon) / 2
folium.Marker(
    location=(mid_lat, mid_lon),
    icon=folium.DivIcon(html=(
        f'<div style="font-size:11px;color:#6a3d9a;font-weight:bold;'
        f'white-space:nowrap">&#8594; +{peak_lag} min</div>')),
).add_to(m)

# Two main markers, colored by the "speed now" snapshot.
folium.CircleMarker([u_lat, u_lon], radius=11, color="#000000", weight=1,
                    fill=True, fill_color=_speed_color(snap[UPSTREAM_SENSOR]),
                    fill_opacity=0.95,
                    tooltip=(f"UPSTREAM {UPSTREAM_SENSOR}: "
                             f"{snap[UPSTREAM_SENSOR]:.0f} mph @ {snap_t}")).add_to(m)
folium.CircleMarker([t_lat, t_lon], radius=13, color="#000000", weight=2,
                    fill=True, fill_color=_speed_color(snap[TARGET_SENSOR]),
                    fill_opacity=0.95,
                    tooltip=(f"TARGET {TARGET_SENSOR}: "
                             f"{snap[TARGET_SENSOR]:.0f} mph @ {snap_t}")).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
print(f"Speed snapshot at {snap_t}  (green>=55, orange>=35, red<35 mph)")
m

### 8b. The honest moment — neighbors help, *and* baseline strength matters

The cross-correlation peak is **real**: upstream speed leads the target, so a spatial model *can* win at short horizons — exactly the band where SARIMA still beats the floor. **But** the *size* of that win depends on how strong your baseline already is (Rodrigues 2022, [DOI:10.1109/TITS.2021.3136141](https://doi.org/10.1109/TITS.2021.3136141)): against a weak baseline a GNN looks heroic; against a well-tuned seasonal model the margin can shrink. State both — that honesty *is* Unit 5's opening: a graph earns its complexity only where a strong baseline leaves signal on the table.

## Where to go next

Open-ended explorations — pick one and run the **direct → interpret → extend** loop:

1. **Aggregate differently.** Flip the config cell to native 5-min (`AGG="5min"`, `SEASON_M=288`, `HORIZONS=[5,15,30,60]`) and re-run. Does the **breakeven horizon** move? Why might coarser aggregation change the floor as well as the model?
2. **Stress the contract.** Find a holiday or an incident window in the series and refit — *where* does SARIMA break, and what **model class** would you reach for when stationarity fails? (Seeds the practice extension.)
3. **Quantify the neighbor.** Beyond cross-correlation, add the upstream sensor as an **exogenous regressor** (`SARIMAX(..., exog=upstream)`) and check whether the 30-min error drops — a hand-built preview of what Unit 5's GNN does network-wide.


---

**Bridge to the supervised practice.** This demo's data is **METR-LA loop detectors** — *already-Eulerian*, fixed sensors that report a speed per location per 5 min. The practice flips the source: **Tel-Aviv bus GPS** (`tlv_all.parquet`) is *Lagrangian* — pings move **with each vehicle** — so you first turn per-vehicle pings into an **Eulerian per-segment speed series** (speed from consecutive ping pairs, pooled over all lines through a corridor) before any of the forecasting above applies. One sign flip matters: in METR-LA `velocity == 0` means **sensor missing** (we replaced it with NaN), but for buses `velocity == 0` means **stopped at a light or dwelling at a stop** — a real signal you must decide to keep or drop, not noise to mask.